# Custom Datasets

This notebook, aimed at instructors, provides instructions on how to obtain data to create your own custom datasets for an asteroid of your choice, should you not want to use the example datasets included or simply wish to create your own. It also includes code tools for nicely formatting the datasets. You should be able to run this notebook in your browser with binder just like the other demos, if you do not wish to install Python. 

For the purposes of this notebook, I will be using the asteroid (617) Patroclus as an example. 

Run the cell below before running anything else!

In [ ]:
import numpy as np
import pandas as pd
import io
import requests
import matplotlib.pyplot as plt
from astropy.io import ascii
from astropy.table import Table, vstack
import astropy.units as u
from astroquery.mpc import MPC
from astroquery.gaia import Gaia
import kete

def kete_ephem(times, obj_name):
    if type(times) == float or type(times) == int:
        times = [times] 
    obj = kete.HorizonsProperties.fetch(obj_name)
    eph = np.zeros((len(times), 4))
    for i in range(len(times)):
        t = times[i].value
        state = kete.propagate_n_body(obj.state, t)
        sun2obs = kete.spice.get_state('Earth', t).pos
        sun2obj = state.pos
        obs2obj = -sun2obs + sun2obj
        # LT correction because I can. velocity is in AU/day, c = 173.14463 AU/day
        sun2obj_lt_sun = kete.propagate_two_body(state, t - sun2obj.r / (173.14463)).pos
        sun2obj_lt_obs = kete.propagate_two_body(state, t - obs2obj.r / (173.14463)).pos
        sun2obs_lt = kete.spice.get_state('Earth', t - obs2obj.r/ (173.14463)).pos
        obs2obj_lt = -sun2obs_lt + sun2obj_lt_obs
        eph[i][0] = t
        eph[i][1] = (-sun2obj_lt_sun).angle_between(-obs2obj_lt)
        eph[i][2] = sun2obj_lt_sun.r
        eph[i][3] = obs2obj_lt.r
        ephemeris = Table(eph, names=['jd', 'phase', 'r', 'delta'])
    return(ephemeris)

# Discovery and Orbits

You can choose to generate discovery datasets using either ATLAS or ZTF (or both?). The availability of data may vary from object to object, so instructions for both are included.


## ATLAS

1. Visit the [CATCH webpage](https://catch.astro.umd.edu/). Make sure "Moving Target" is selected. Enter your object name in the search box and select the correct object. Use the filters panel and make sure only ATLAS Haleakela, ATLAS Mauna Loa, ATLAS Rio Hurtado, ATLAS Sutherland are checked under the "Data Sources" panel. Click 'Start' and wait for the search to run (usually a few minutes).
2. Click the date column to sort by date. Search for groups of 3 or more images taken in a single night, for three or four nights over at least a three month span. Ensure the asteroid is visible by looking at the preview of the data and checking the V magnitude column (V < 19 means the asteroid should be bright enough, this is the ATLAS median detection limit). Select all the images you want to use with the checkboxes and click "Download" to obtain a ZIP file.
3. Uncompress the ZIP file. Inside the fits folder are the fits images that you can provide to your class as-is, and you can optionally stop here. However, these images are cropped to be centered on the asteroid you chose. For a more authentic asteroid hunting experience, I like generate new images that are not centered on the asteroid, generating the appropriate urls using the following python script. Place the csv from the ZIP file into the same folder as this notebook, then substitute '617_data.csv' for the your filename and 'Mystery Asteroid Discovery.csv' for wherever you want to save the list of urls to.

To obtain **difference images** (especially helpful for the comets exercises) from ATLAS, simply change the ``.fits`` in each generated url to ``.diff``.

In [ ]:
def get_urls_atlas(path, name):
    urls = []
    tab = ascii.read(path)
    tab.sort(['mjd_start'])
    starts = np.where(np.abs(tab['mjd_start'] - np.roll(tab['mjd_start'], 1)) > 0.75)[0]
    np.append(starts, 0)
    for i in range(len(starts)-1):
        sub_tab = tab[starts[i]:starts[i+1]]
        ra = sub_tab[0]['ra']
        dec = sub_tab[0]['dec']
        for j in range(len(sub_tab)):
            urls.append(sub_tab[j]['archive_url'] + f'?ra={ra}&dec={dec}&size=0.3deg&format=fits')
    t = Table([urls], names = [name])
    ascii.write(t, name+'.csv', format = 'csv')

#change me!
get_urls_atlas('617_data.csv', 'Mystery Asteroid Discovery ATLAS')

## ZTF

1. Visit the [ZTF Archive](https://irsa.ipac.caltech.edu/applications/ztf/) and select the "Search by Solar System Object/Orbit" tab. Enter your object name in the search box, select the correct object, click "Search".
2. Click the 'obsdate' column to sort by date. Search for groups of 2 or more images taken in a single night, for three or four nights over at least a three month span. Ideally, these two images should be in the same filter, but this doesn't have to be the case (it is possible some stars with weird colors are only visible in one filter and thus might be confused with the asteroid, but I haven't seen this happen in practice). The preview images on the right can be helpful, as they show all positions the object was found by ZTF (yellow shows the detection of the current selection, and red other detections retrieved by your search query). Since asteroids imaged on the same night will show up in the same field of view, this makes identifying image pairs (or triplets) that show the asteroid's non-sideral motion very obvious at a glance. Ensure the asteroid is visible using the previews, then use the checkboxes in the very left hand column to select which images you would like to download.
3. Click the filter icon at the top of the checkbox column to filter for only those files which you have selected, then click the save icon to save a summary of the table to your computer. Click save to save and download the file.
4. Place the csv you downloaded into the same folder as this notebook. Run the following script to generate a list of urls from which the individual science images can be downloaded. As with the ATLAS script, these new images are not necessarily centered on the asteroid for a more authentic asteroid hunting experience.

To obtain **difference images** (especially helpful for the comets exercises) from ZTF, simply change the ``sciimg.fits`` in each url to ``scimrefdiffimg.fits.fz``.


In [ ]:
def get_urls_ztf(path, name):
    urls = []
    table = ascii.read('path')
    table['round'] = np.round(np.asarray(table['obsjd']))
    nights = np.unique(table['round'])
    for night in nights:
        f_n = table['round'] == night
        table_n = table[f_n]
        ra = str(table_n[0]['ra_obj'])
        dec = str(table_n[0]['dec_obj'])
        cutout = '?center='+ ra + ',' + dec + '&size=500arcsec&gzip=false'
        for obs in table_n:
            year = str(obs['filefracday'])[0:4]
            month = str(obs['filefracday'])[4:6]
            day = str(obs['filefracday'])[6:8]
            fracday = str(obs['filefracday'])[8:]
            filefracday = str(obs['filefracday'])
            paddedfield = ('000000' + str(obs['field']))[-6:]
            filtercode = obs['filtercode']
            paddedccdid = ('000000' + str(obs['ccdid']))[-2:]
            imgtypecode = obs['imgtypecode']
            qid = str(obs['qid'])
            url = ('https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci/'+year+'/'+month+day+'/'+fracday+'/ztf_'+filefracday+'_'+paddedfield+'_'+filtercode+'_c'+paddedccdid+'_'+imgtypecode+'_q'+qid+'_sciimg.fits' + cutout)
            urls.append(url)
    t = Table(urls, names = ['urls'])
    ascii.write(t, name + '.csv', format = 'csv')
            
#change me!
get_urls_ztf('table_617-Patroclus20000617.csv', 'Mystery Asteroid Discovery ATLAS')

# Phase Function and Absolute Magnitude

You can choose to generate phase function datasets using either ATLAS or ZTF (or both?). The availability of data may vary from object to object, so instructions for both are included.

## ATLAS

Using astroquery, it is easy to grab ATLAS observations that have been submitted to the [Minor Planet Center](https://www.minorplanetcenter.net/). Use the plot to check that there are enough observations present in ATLAS to construct a reasonable phase curve. Here, we are filtering only for those observations taken with the "o" filter--this is close enough to V for the diameter calculation to work, and there tend to be more datapoints in the "o" filter anyways. 

After obtaining the observations, we need to add ephemeris information (heliocentric distance, geocentric distance, phase). This is relatively easy to do with kete. Rather than specify a particular observatory, we just use the geocenter as our observer, the difference for our purposes is negligible. 

Replace '617' with the number of your object, and replace 'Mystery Asteroid ATLAS Magnitudes.csv' with your desired filename.

In [ ]:
# change me!
obj = '617'

obs = MPC.get_observations(obj)
atlas_codes = ['M22', 'T05', 'T08', 'W68']


in_atlas = np.zeros(len(obs))
for a in atlas_codes:
    f = np.logical_and(obs['observatory'] == a, obs['band'] == 'o')
    in_atlas += f
in_atlas = in_atlas == 1.

obs_atlas = obs[in_atlas]

plt.title('ATLAS magnitudes')
plt.plot(obs_atlas['epoch'], obs_atlas['mag'], '.')
plt.ylabel('Magnitude')
plt.xlabel('Epoch (jd)')
plt.show()

eph = kete_ephem(obs_atlas['epoch'],obj)

eph['mag'] = obs_atlas['mag']

# change me!
ascii.write(eph, "Mystery Asteroid ATLAS Magnitudes.csv", format = 'csv')

## ZTF

Similarly, we can grab the ZTF magnitudes for an object if we wish--this time using the [fink](https://fink-broker.org/) API. Fink helpfully has a routine to query the ephemerides built in! We will filter for observations in the 'g' filter for ZTF, which is roughly equivalent to V. Again, check that there are a sufficient number of data points using the plot--some objects will have better coverage in ATLAS than ZTF or vice versa. As you can see, Patroclus has enough points in both surveys.

As with ATLAS, we will obtain ephemeris information and write the results to a file. Replace '617' with the number of your object, and replace 'Mystery Asteroid ZTF Magnitudes.csv' with your desired filename.

In [ ]:
#change me!
obj = 617

r = requests.post(
"https://api.ztf.fink-portal.org/api/v1/sso",
json ={
  "n_or_d": obj,
  "withEphem": True,
  "withResiduals": False,
  "withCutouts": False,
  "cutout-kind": "Science",
  "columns": "i:jd,i:magpsf,i:fid,i:ssnamenr",
  "output-format": "json"
}
)


obs_ztf = pd.read_json(io.BytesIO(r.content))
obs_ztf = obs_ztf[obs_ztf['i:fid'] == 1]

plt.title('ZTF magnitudes')
plt.plot(obs_ztf['Date'], obs_ztf['i:magpsf'], '.')
plt.ylabel('Magnitude')
plt.xlabel('Epoch (jd)')
plt.show()

obs_ztf[['Date', 'i:magpsf', 'Phase', 'Dhelio', 'Dobs']]
obs_ztf.rename(columns={"Date":"jd", "i:magpsf": "mag", "Dhelio": "r", "Dobs":"delta"}, inplace = True)

#change me!
obs_ztf[["jd", "Phase", "r", "delta", "mag"]].to_csv("Mystery Asteroid ZTF Magnitudes.csv", index = False)

# Lightcurves

There are several options for retrieving lightcurves. Note that ALCDEF records light curves in magnitudes, while DAMIT uses fluxes. In practice either method works since magnitude differences can be converted to flux ratios, but this is something to pay attention to when estimating the asteroid's axis ratios.

## Asteroid Light Curve Database (ALCDEF)

1. Visit [ALCDEF](https://alcdef.org/alcdef.php). Click on search database, enter the name or number of your object, and click submit. The 'LCDB Only' option should be *unchecked*.
2. In the right panel, examine the search results and choose the correct asteroid using the radio button, then click display.
3. Open the 'Multi-LC Tool' (opens in a new tab) to display and examine multiple lightcurves in the same plot. Select a series of lightcurves taken in the same (or similar) filters over the span of a few days or weeks using the checkboxes. Click plot to visually examine the light curves you selected for signs of periodicity and to make sure you have enough datapoints.
4. When you are satisfied with your choice, return to the original tab and select only the light curves you identified with the 'Multi-LC Tool' with the checkboxes and click 'Download'
5. Place the file you downloaded into the same folder as this notebook. Run the following script to nicely format the data file.

In [ ]:
#change me!
fname = 'ALCDEF_617_Patroclus_20260922_092100.txt'

f1 = open(fname,'r')
lines = np.array(f1.readlines())
f1.close()

s_index = np.where(lines == 'ENDMETADATA\n')[0]+1
e_index = np.where(lines == 'STARTMETADATA\n')[0][1:]-2
e_index = np.append(e_index, -1)
s_index, e_index

t_final = Table([], names = ['mag', 'mag err', 'jd'])

for i in range(len(s_index)):
    t = ascii.read(fname, data_start = s_index[i], data_end = e_index[i], names = ['date', 'mag', 'mag err'])
    t['jd'] = [float(j[5:]) for j in t['date']]
    t_final = vstack([t_final, t['mag', 'mag err', 'jd']])

ascii.write(t_final, 'Mystery Asteroid Lightcurves.csv', format='csv')

## Asteroid Light Curve Database via the PDS Small Bodies Node


1. Visit the [PDS Small Bodies Node's page for the Asteroid Light Curve Database (ALCDEF)](https://sbn.psi.edu/pds/resource/alcdef.html). Click `Browse` then `data`.
2. Identify the folder corresponding to your asteroid, it will be the closest round number to your asteroid's number without going over, so for (617) Patroclus, it would be 600.
3. Enter that folder and download a file with a name like `	alcdef_lcdata-xxx-xxx.csv`. For Patroclus, this is `alcdef_lcdata-600-699.csv`
4. Place the csv file you downloaded into the same folder as this notebook. Run the following script to check for lightcurves of your asteroid.
5. Using the plot below, identify short (days to weeks) long blocks of light curves of your asteroid to select a portion of the light curve with dense coverage. Adjust the t_min and t_max variable to include only those observations and output the results to a file.

In [ ]:
#change me!
obj = 617

alcdef = ascii.read('alcdef_lcdata-600-699.csv', data_start = 21, names=['n', 'name', 'record', 'JD', 'mag', 'mag err'])
alcdef = alcdef[alcdef['n'] == obj]

plt.title('All Lightcurves') 
plt.plot(alcdef['JD'], alcdef['mag'], '.')
plt.ylabel('Magnitude')
plt.xlabel('JD')
plt.show()

#change me!
t_min = 2455800
#change me!
t_max = 2455850

alcdef = alcdef[np.logical_and(alcdef['JD'] > t_min, alcdef['JD'] < t_max)]

plt.title('Selected Lightcurves') 
plt.plot(alcdef['JD'], alcdef['mag'], '.')
plt.ylabel('Magnitude')
plt.xlabel('JD')
plt.show()

#change me!
ascii.write(alcdef['JD', 'mag', 'mag err'], "Mystery Asteroid Lightcurves.csv", format = 'csv')

## DAMIT

Some asteroids have a full shape model and spin solution published in the Database of Asteroid Models from Inversion Techniques (DAMIT). Unfortunately, (617) Patroclus is not one of those asteroids--but the one you pick might be!

1. Visit [DAMIT](https://damit.cuni.cz/projects/damit/) and search for your asteroid.
2. If a model exists, it will show up in search. Click 'lc.txt' to open a list of all the light curves. (Hint: under the 'More' column you can click the three dots for any model and then click 'Model Fit to Lightcurves' tto see all the light curve data (blue points) compared to the model prediction (red lines). This is a very easy way to check if the asteroid has at least one dense-in-time light curve before downloading the data).
3. Save the file 'lc.txt' to the same folder as this notebook. Run the following script to check for lightcurves of your asteroid.
4. Using the plot below, identify short (days to weeks) long blocks of light curves of your asteroid to select a portion of the light curve with dense coverage. Adjust the t_min and t_max variable to include only those observations and output the results to a file.

In [ ]:
#change me!
fname = 'A3414.lc.txt'

f1 = open(fname,'r')
lines = np.array(f1.readlines())
f1.close()

jd = []
flux = []
for l in lines:
    if len(l) > 10:
        jd.append(float(l.split(' ')[0]))
        flux.append(float(l.split(' ')[2]))
        
damit = Table([jd, flux], names = ['jd', 'flux'])

plt.title('All Lightcurves') 
plt.plot(damit['jd'], damit['flux'], '.')
plt.ylabel('Magnitude')
plt.xlabel('JD')
plt.show()

#change me!
t_min = 2442750
#change me!
t_max = 2442760

damit = damit[np.logical_and(damit['jd'] > t_min, damit['jd'] < t_max)]

plt.title('Selected Lightcurves') 
plt.plot(damit['jd'], damit['flux'], '.')
plt.ylabel('Magnitude')
plt.xlabel('JD')
plt.show()

#change me!


# Diameter and Albedo

1. Access the [Small Bodies Near and Far (SBNAF)](https://ird.konkoly.hu/) database. Click on the SQL search tab and input the following query (modified to include the name of your asteroid, of course!)

   
    ```targetname LIKE '%Patroclus%' AND ( observatory_project ILIKE 'WISE' OR observatory_project ILIKE 'AKARI' ) AND ( band_filter ILIKE 'L18W' OR band_filter ILIKE 'S9W' OR band_filter ILIKE 'W4' OR band_filter ILIKE 'W3' )```
2. In the upper left, click on the checkboxes icon and
    - **Uncheck** the following boxes: `naifid` and `targetname`. This anonymizes your asteroid. 
    - **Check** the following boxes: `datetime`, `band_filter`, `heliiocentric_distance_r`, `obscentric_distance_delta`, and `phase_angle_alpha`. This adds important metadata, including the observing geometry information for this asteroid via an ephemeris.
3. Click the save to disk icon and save the table as a CSV. This can be provided directly to students as is, or lightly edited to only contain observations from a single epoch.

TBD: TAP query access?

# Spectroscopy

Use the plot function to check if your object is in the [Gaia DR3](https://www.cosmos.esa.int/web/gaia/data-release-3). The Gaia DR3 actually returns instrument and solar corrected spectra, so this tool will both fetch and *add back in* the instrumental effects and solar spectrum (My rationale for this step: I find that adding the extra correction step helps to differentiate conceptually between flux, which is measured at a telescope, and reflectance, which is a derived ratio of recieved vs. reflected light). 

In [ ]:
#change me!
obj = 617

Gaia.MAIN_GAIA_TABLE = "gaiadr3.sso_reflectance_spectrum"

query = """SELECT gaiadr3.sso_reflectance_spectrum.number_mp, gaiadr3.sso_reflectance_spectrum.denomination, gaiadr3.sso_reflectance_spectrum.wavelength, gaiadr3.sso_reflectance_spectrum.reflectance_spectrum, gaiadr3.sso_reflectance_spectrum.reflectance_spectrum_err, gaiadr3.sso_reflectance_spectrum.reflectance_spectrum_flag
            FROM gaiadr3.sso_reflectance_spectrum
            WHERE gaiadr3.sso_reflectance_spectrum.reflectance_spectrum_flag = 0 AND gaiadr3.sso_reflectance_spectrum.number_mp = """ + str(obj)

job = Gaia.launch_job(query)
obs_gaia = job.get_results()

plt.title('Gaia Spectrum')
plt.plot(obs_gaia['wavelength'], obs_gaia['reflectance_spectrum'])
plt.ylabel('Reflectance')
plt.xlabel('Wavelength (nm)')
plt.show()

# or retrieve this file from the Asteroids Exercises folder and place in the same directory as this notebook
solar_16 = ascii.read('../Asteroids Exercises/gaia_solar.txt')


flux = obs_gaia['reflectance_spectrum'] * solar_16['Normalized Flux']
flux_err = obs_gaia['reflectance_spectrum_err'] * solar_16['Normalized Flux']
flag = obs_gaia['reflectance_spectrum_flag']

ascii.write(Table([g_wave, flux, flux_err, flag], names = ['Wavelength (nm)', 'Normalized Flux', 'Flux Error', 'Quality Flag']), \
            #change me!
            'Mystery Asteroid Gaia Spectrum.csv', format = 'csv')

